# TRACE Reviewer-Revision Companion Notebook

**Purpose:** Reproduce reviewer-requested robustness checks and revised figures from the submitted TRACE experiment, its archived outputs, and the original datasets.  
**Scope:** This companion notebook does not replace the main TRACE pipeline. It reuses the validation-locked experimental design and never selects a new primary configuration from held-out test performance.

### What this notebook produces

| Reviewer comment | Notebook action | New training? |
|---|---|---:|
| R1-1 Causal terminology | Generate replacement text and insertion locations | No |
| R1-2 Nighttime-zero fairness | Recompute scope-specific metrics from identical saved test predictions | No |
| R1-3 Hyperparameter/capacity | Extract validation ranges and checkpoint-derived model capacity | No |
| R1-4 Overclaim in Abstract | Generate qualified replacement wording | No |
| R1-5 Figure 6/7 overlap | Re-export SHAP waterfalls from the saved TRACE bundles | **No** |
| R2-1 0.75/0.25 weighting | validation-only selection weight sensitivity | **Yes, TRACE only** |
| R2-2 Figure 2 issue time | Replot a complete next-calendar-day trajectory | No |
| R2-3 redundancy/resolution | Produce horizon-only Figure 3 and audit Figure 4 source resolution | No |

> **Important:** `0.75 RMSE + 0.25 MAE` is a **validation-selection score**, not the training loss. The sensitivity analysis is a reviewer-requested robustness check; it must not be used to choose a new primary setting from test performance.

### Methodological references checked for this revision
- Forecasting: Principles and Practice — rolling forecasting-origin evaluation uses only observations available before each forecast origin.
- scikit-learn `TimeSeriesSplit` — time-series evaluation should preserve chronology rather than shuffle future observations into training.
- SHAP waterfall documentation — `max_display` controls how many local contributions are shown; reducing it is an appropriate display-only fix for dense annotations.

In [ ]:
# ============================================================
# USER CONFIG — review this cell, then use Run All.
# ============================================================
import os

# Reviewer 2 Comment 1: genuine sensitivity rerun.
# Environment variable TRACE_RUN_SENSITIVITY=0 can disable it for a quick check.
RUN_WEIGHT_SENSITIVITY = os.environ.get("TRACE_RUN_SENSITIVITY", "1") == "1"

# Reviewer 1 Comment 5: re-export local SHAP plots from the already fitted saved bundles.
RUN_SHAP_REEXPORT = os.environ.get("TRACE_RUN_SHAP", "1") == "1"

# Original 0.75 RMSE + 0.25 MAE is already in the submitted experiment outputs.
REUSE_ORIGINAL_025 = True
# Values below are MAE weights. Thus 0.50 = 50/50 and 0.75 = 25/75 RMSE/MAE.
SENSITIVITY_MAE_WEIGHTS = [0.50, 0.75]

# Figure settings
FIGURE_DPI = 600
SHAP_MAX_DISPLAY = 11
SHAP_FIGSIZE = (14, 8)
SHAP_ANNOTATION_FONTSIZE = 9

# Do not use held-out test performance to select a new weighting.
ALLOW_TEST_BASED_WEIGHT_SELECTION = False
assert not ALLOW_TEST_BASED_WEIGHT_SELECTION, "Test-based weight selection is intentionally prohibited."

In [ ]:
# ============================================================
# Imports, package root detection, output folders
# ============================================================
from pathlib import Path
import dataclasses
import hashlib
import io
import json
import os
import shutil
import sys
import tempfile
import time
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = str


def find_package_root():
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "code" / "TRACE_Reviewer_Revision_AllInOne.ipynb").exists() and (p / "README.md").exists():
            return p
    raise FileNotFoundError(
        "Could not locate the TRACE revision package root. "
        "Open this notebook from inside the cloned TCFN4PVForecasting repository."
    )

ROOT = find_package_root()
INPUT_DIR = ROOT / "input"
ZIP_CANDIDATES = [
    INPUT_DIR / "TRACE_PV24_outputs.zip",
    ROOT / "TRACE_PV24_outputs.zip",
    ROOT.parent / "TRACE_PV24_outputs.zip",
]
ZIP_PATH = next((p for p in ZIP_CANDIDATES if p.exists()), ZIP_CANDIDATES[0])
MANUSCRIPT_PATH = INPUT_DIR / "TRACE_submission_manuscript.docx"
if not ZIP_PATH.exists():
    raise FileNotFoundError(
        "TRACE_PV24_outputs.zip was not found. Put the original ZIP in package/input/, "
        "the package root, or the parent folder, then Run All again."
    )
OUTPUT_ROOT = ROOT / "outputs"
OUT_SCOPE = OUTPUT_ROOT / "01_scope_benchmark"
OUT_SPECS = OUTPUT_ROOT / "02_model_specs"
OUT_FIG = OUTPUT_ROOT / "03_revised_figures"
OUT_SHAP = OUTPUT_ROOT / "04_shap_reexport"
OUT_SENS = OUTPUT_ROOT / "05_weight_sensitivity"
OUT_SUMMARY = OUTPUT_ROOT / "06_summary"
WORK_DIR = ROOT / "_working_core"
for d in [OUT_SCOPE, OUT_SPECS, OUT_FIG, OUT_SHAP, OUT_SENS, OUT_SUMMARY, WORK_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SITES = ["Dangjin", "Gwangyang"]
print("PACKAGE ROOT:", ROOT)
print("INPUT ZIP:", ZIP_PATH)
print("MANUSCRIPT:", MANUSCRIPT_PATH)
print("Python:", sys.version.split()[0])

In [ ]:
# ============================================================
# Input integrity + archive inventory
# ============================================================
def sha256(path, block=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(block)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

assert ZIP_PATH.exists(), ZIP_PATH
# The manuscript is optional and is used only for provenance hashing when supplied locally.

with zipfile.ZipFile(ZIP_PATH) as z:
    names = z.namelist()
    required_suffixes = [
        "TRACE_PV24_Guarded_Dual_Strategy_Ensemble_Windows_Jupyter.ipynb",
        "Dangjin_Landfill_PV_Dataset.csv",
        "Gwangyang_Port_Site2_PV_Dataset.csv",
        "TRACE_PV24_final_results.csv",
        "Dangjin_all_models_paired_test_predictions.csv",
        "Gwangyang_all_models_paired_test_predictions.csv",
    ]
    missing = [s for s in required_suffixes if not any(n.endswith(s) for n in names)]
    if missing:
        raise FileNotFoundError(f"Archive is missing required members: {missing}")

integrity_rows = [
    {"file": str(ZIP_PATH.relative_to(ROOT)), "bytes": ZIP_PATH.stat().st_size, "sha256": sha256(ZIP_PATH)},
]
if MANUSCRIPT_PATH.exists():
    integrity_rows.append({"file": str(MANUSCRIPT_PATH.relative_to(ROOT)), "bytes": MANUSCRIPT_PATH.stat().st_size, "sha256": sha256(MANUSCRIPT_PATH)})
integrity = pd.DataFrame(integrity_rows)
display(integrity)
integrity.to_csv(OUT_SUMMARY / "input_integrity.csv", index=False)
print(f"Archive members: {len(names)}")

In [ ]:
# ============================================================
# Shared ZIP helpers and metrics
# ============================================================
def zip_member_by_suffix(z, suffix):
    hits = [n for n in z.namelist() if n.endswith(suffix)]
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected exactly one '*{suffix}', found: {hits}")
    return hits[0]


def read_csv_suffix(z, suffix, **kwargs):
    return pd.read_csv(z.open(zip_member_by_suffix(z, suffix)), **kwargs)


def read_bytes_suffix(z, suffix):
    return z.read(zip_member_by_suffix(z, suffix))


def rmse(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    return float(np.sqrt(np.mean((y-p)**2)))


def mae(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    return float(np.mean(np.abs(y-p)))


def r2(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    den = np.sum((y-y.mean())**2)
    return float(1 - np.sum((y-p)**2)/den) if den > 0 else np.nan

## 1. Reviewer 1 Comment 2 — nighttime-zero fairness

All benchmark models are reevaluated over the **same held-out origin–horizon pairs** using the following scopes:

- Overall
- Solar-eligible: targets outside TRACE's structural-zero mask
- Actual-positive
- Actual-zero

This step **does not retrain any model**. It directly tests, from the saved predictions, whether TRACE's treatment of nighttime zeros alone explains its relative benchmark position.

In [ ]:
# ============================================================
# R1 Comment 2 — scope-specific benchmark
# ============================================================
rows = []
with zipfile.ZipFile(ZIP_PATH) as z:
    for site in SITES:
        pair = read_csv_suffix(z, f"{site}_all_models_paired_test_predictions.csv")
        tr = read_csv_suffix(z, f"{site}_TRACE_PV24_test_predictions.csv")
        keys = ["forecast_origin", "target_time", "horizon", "actual"]
        aux = tr[keys + ["structural_zero", "actual_active"]].copy()
        df = pair.merge(aux, on=keys, how="left", validate="one_to_one")
        if df[["structural_zero", "actual_active"]].isna().any().any():
            raise RuntimeError(f"{site}: TRACE masks failed to align with paired predictions")

        scopes = {
            "Overall": np.ones(len(df), dtype=bool),
            "Solar-eligible": ~df["structural_zero"].astype(bool).to_numpy(),
            "Actual-positive": df["actual_active"].astype(bool).to_numpy(),
            "Actual-zero": ~df["actual_active"].astype(bool).to_numpy(),
        }
        models = [c for c in pair.columns if c not in keys]
        for scope, mask in scopes.items():
            y = df.loc[mask, "actual"].to_numpy(float)
            for model in models:
                p = df.loc[mask, model].to_numpy(float)
                rows.append({
                    "site": site, "scope": scope, "model": model, "n": len(y),
                    "RMSE": rmse(y,p), "MAE": mae(y,p), "R2": r2(y,p),
                })

scope_results = pd.DataFrame(rows)
scope_results["RMSE_rank"] = scope_results.groupby(["site","scope"])["RMSE"].rank(method="min")
scope_results["MAE_rank"] = scope_results.groupby(["site","scope"])["MAE"].rank(method="min")
scope_results.to_csv(OUT_SCOPE / "R1_scope_benchmark_all_models.csv", index=False)

restricted = scope_results[scope_results.scope.isin(["Solar-eligible","Actual-positive"])].copy()
restricted = restricted.sort_values(["site","scope","RMSE"])
restricted.to_csv(OUT_SCOPE / "R1_scope_benchmark_restricted.csv", index=False)

display(restricted.groupby(["site","scope"], group_keys=False).head(5))

In [ ]:
# Reviewer-ready compact evidence for R1-C2
r1c2_evidence = []
for site in SITES:
    for scope in ["Solar-eligible", "Actual-positive"]:
        s = restricted[(restricted.site==site) & (restricted.scope==scope)].sort_values("RMSE")
        best = s.iloc[0]
        trace = s[s.model=="TRACE-PV24"].iloc[0]
        r1c2_evidence.append({
            "site": site,
            "scope": scope,
            "best_RMSE_model": best.model,
            "best_RMSE": best.RMSE,
            "TRACE_RMSE": trace.RMSE,
            "TRACE_MAE": trace.MAE,
            "TRACE_RMSE_rank": trace.RMSE_rank,
            "TRACE_MAE_rank": trace.MAE_rank,
        })
r1c2_evidence = pd.DataFrame(r1c2_evidence)
display(r1c2_evidence)
r1c2_evidence.to_csv(OUT_SCOPE / "R1_comment2_reviewer_ready_evidence.csv", index=False)

## 2. Reviewer 1 Comment 3 — hyperparameter ranges and neural model capacity

The tables are extracted directly from the existing `*_validation_search.csv` files and representative-seed checkpoints.  
No new fitting is performed.

In [ ]:
# ============================================================
# R1 Comment 3 — TRACE selected hyperparameters + neural specs
# ============================================================
EXPERTS = ["extra_cls", "hist_cls", "extra_reg", "hist_reg"]
ARCH = {
    "RNN":"1 recurrent layer; 128 hidden units; Linear–SiLU–Linear 24-step head",
    "LSTM":"1 LSTM layer; 128 hidden units; Linear–SiLU–Linear 24-step head",
    "GRU":"1 GRU layer; 128 hidden units; Linear–SiLU–Linear 24-step head",
    "BiLSTM":"1 bidirectional LSTM layer; 128 hidden units; Linear–SiLU–Linear 24-step head",
    "1D-CNN":"1 Conv1d block + batch normalization/pooling; 128-unit head",
    "1D-CNN-BiLSTM":"1 Conv1d block + batch normalization/pooling + 1 BiLSTM; 128-unit head",
    "Attn-GRU":"1 GRU + 4-head multi-head attention + LayerNorm; 128-unit head",
    "TCN":"3 residual causal/dilated blocks (dilations 1,2,4); 128-unit head",
    "Transformer":"1 four-head context-attention block + feed-forward/norms; 128-unit head",
    "TCFN":"trend Conv1d + 4-head attention context + fusion LSTM; scenario-specific units",
    "TCFN-168":"same TCFN architecture with 168-h lookback; scenario-specific units",
}
LOOKBACK = {m:24 for m in ARCH}
LOOKBACK["TCFN-168"] = 168
CANDIDATE_RANGES = [
    ("ExtraTrees n_estimators","{120,160,220}","base=160"),
    ("ExtraTrees max_depth","{14,18,24}","base=20"),
    ("ExtraTrees min_samples_leaf","integer 2–15","base=5"),
    ("ExtraTrees min_samples_split","integer 2–14","base=4"),
    ("ExtraTrees max_features","Uniform[0.45,0.90]","base=0.70"),
    ("HistGB learning_rate","log-uniform [0.025,0.12]","base=0.06"),
    ("HistGB max_iter","{160,240,340}","base=250"),
    ("HistGB max_leaf_nodes","{15,31,63}","base=31"),
    ("HistGB min_samples_leaf","integer 20–100","base=40"),
    ("HistGB l2_regularization","log-uniform [0.001,20]","base=1"),
    ("HistGB max_bins","{127,255}","base=255"),
    ("Hurdle alpha/beta","{0,0.25,0.50,0.75,1}","validation grid"),
    ("Hurdle threshold tau","{0.25,0.35,0.45,0.55,0.65}","validation grid"),
]


def torch_checkpoint_info(raw):
    try:
        import torch
    except Exception as e:
        return {"parameter_count": np.nan, "checkpoint_note": f"torch unavailable: {e}"}
    ck = torch.load(io.BytesIO(raw), map_location="cpu", weights_only=False)
    state = None
    if isinstance(ck, dict):
        for key in ["model_state_dict", "state_dict"]:
            if key in ck and isinstance(ck[key], dict):
                state = ck[key]; break
        if state is None and ck and all(hasattr(v, "numel") for v in ck.values()):
            state = ck
    if state is None:
        return {"parameter_count": np.nan}
    cnt = sum(int(v.numel()) for k,v in state.items()
              if not any(x in k for x in ["running_mean","running_var","num_batches_tracked"]))
    meta = {"parameter_count": cnt}
    if isinstance(ck, dict):
        for k in ["epoch","best_epoch","lookback","units","dropout","learning_rate","lr","heads","scenario","scenario_name"]:
            if k in ck:
                meta[k] = ck[k]
        for kk in ["config","params","model_params","hparams"]:
            if kk in ck and isinstance(ck[kk], dict):
                for k,v in ck[kk].items():
                    if k in ["epoch","best_epoch","lookback","units","dropout","learning_rate","lr","heads","scenario","scenario_name"]:
                        meta[k] = v
    return meta

selected_rows, spec_rows = [], []
with zipfile.ZipFile(ZIP_PATH) as z:
    for site in SITES:
        for expert in EXPERTS:
            df = read_csv_suffix(z, f"{site}_{expert}_validation_search.csv").sort_values("selection_score")
            r = df.iloc[0].to_dict()
            r = {k:v for k,v in r.items() if k not in ["site","expert"]}
            selected_rows.append({"site":site,"expert":expert,**r})
        for model in ARCH:
            suffix = f"{site}_{model}_representative_seed_checkpoint.pt"
            hits = [n for n in z.namelist() if n.endswith(suffix)]
            meta = torch_checkpoint_info(z.read(hits[0])) if hits else {"parameter_count":np.nan}
            spec_rows.append({"site":site,"model":model,"lookback_h":LOOKBACK[model],"architecture":ARCH[model],**meta})

selected_hparams = pd.DataFrame(selected_rows)
candidate_ranges = pd.DataFrame(CANDIDATE_RANGES, columns=["parameter","candidate_range","note"])
neural_specs = pd.DataFrame(spec_rows)
selected_hparams.to_csv(OUT_SPECS / "R1_TRACE_selected_hyperparameters.csv", index=False)
candidate_ranges.to_csv(OUT_SPECS / "R1_TRACE_candidate_ranges.csv", index=False)
neural_specs.to_csv(OUT_SPECS / "R1_neural_model_specs.csv", index=False)

display(candidate_ranges)
display(selected_hparams[[c for c in ["site","expert","trial","RMSE","MAE","selection_score"] if c in selected_hparams.columns]])
display(neural_specs[[c for c in ["site","model","lookback_h","parameter_count","best_epoch","epoch"] if c in neural_specs.columns]])

## 3. Reviewer 2 Comments 2/3 — Figure 2 and Figure 3 replacements

- Figure 2 avoids the earlier mid-day issue-time examples and displays a complete profile from a **23:00 forecast origin to 00:00–23:00 on the next calendar day**.
- Figure 3 removes the aggregate ranking already reported in Table 9 and retains **horizon-wise RMSE only**.
- Figure 4 audits pixel and DPI metadata for the original permutation-XAI PNG files, separating source-resolution handling from model reruns.

In [ ]:
# ============================================================
# R2 Comments 2/3 — revised figures + Figure 4 source audit
# ============================================================
FIG3_MODELS = ["TRACE-PV24", "1D-CNN-BiLSTM", "RNN", "TCFN"]
fig2_manifest = []
fig4_audit = []

with zipfile.ZipFile(ZIP_PATH) as z:
    for site in SITES:
        df = read_csv_suffix(z, f"{site}_all_models_paired_test_predictions.csv")
        df["forecast_origin"] = pd.to_datetime(df["forecast_origin"])
        df["target_time"] = pd.to_datetime(df["target_time"])

        uniq = df[["target_time","actual"]].drop_duplicates("target_time").copy()
        uniq["day"] = uniq.target_time.dt.floor("D")
        daily = (uniq.groupby("day")
                 .agg(hours=("target_time","size"), energy=("actual","sum"))
                 .query("hours == 24")
                 .sort_values("energy", ascending=False))
        chosen = None
        for day in daily.index:
            origin = day - pd.Timedelta(hours=1)
            g = df[df.forecast_origin == origin].sort_values("horizon")
            if len(g)==24 and g.target_time.min()==day and g.target_time.max()==day+pd.Timedelta(hours=23):
                chosen = (day, origin, g)
                break
        if chosen is None:
            raise RuntimeError(f"{site}: no complete 23:00-origin next-day example found")
        day, origin, g = chosen

        fig, ax = plt.subplots(figsize=(10.5,5.5))
        ax.plot(g.target_time, g.actual, label="Actual", linewidth=2)
        ax.plot(g.target_time, g["TRACE-PV24"], label="TRACE", linewidth=2)
        ax.plot(g.target_time, g["1D-CNN-BiLSTM"], label="1D-CNN–BiLSTM", linewidth=1.4)
        ax.set_title(f"{site}: complete next-day profile (origin {origin:%Y-%m-%d %H:%M})")
        ax.set_xlabel("Target time")
        ax.set_ylabel("PV power (kW)")
        ax.legend()
        fig.autofmt_xdate()
        fig.tight_layout()
        f2 = OUT_FIG / f"Figure2_{site}_complete_next_day.png"
        fig.savefig(f2, dpi=FIGURE_DPI, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        fig2_manifest.append({"site":site,"forecast_origin":origin,"target_day":day,"target_energy":float(g.actual.sum()),"file":str(f2.relative_to(ROOT))})

        hrows=[]
        for model in FIG3_MODELS:
            for h, x in df.groupby("horizon"):
                hrows.append({"model":model,"horizon":int(h),"RMSE":rmse(x.actual, x[model])})
        hd = pd.DataFrame(hrows)
        fig, ax = plt.subplots(figsize=(9,5.2))
        for model, x in hd.groupby("model"):
            ax.plot(x.horizon, x.RMSE, marker="o", markersize=3, label=model)
        ax.set_xlabel("Forecast horizon (h)")
        ax.set_ylabel("RMSE (kW)")
        ax.set_title(f"{site}: horizon-wise RMSE")
        ax.set_xticks([1,4,8,12,16,20,24])
        ax.legend()
        fig.tight_layout()
        f3 = OUT_FIG / f"Figure3_{site}_horizon_RMSE_only.png"
        fig.savefig(f3, dpi=FIGURE_DPI, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        hd.to_csv(OUT_FIG / f"Figure3_{site}_horizon_RMSE.csv", index=False)

        # Figure 4 original source image metadata audit.
        try:
            from PIL import Image
            raw = read_bytes_suffix(z, f"{site}_TRACE_PV24_family_permutation_XAI.png")
            source_out = OUT_FIG / f"Figure4_{site}_family_permutation_XAI_600dpi_source.png"
            source_out.write_bytes(raw)
            img = Image.open(io.BytesIO(raw))
            fig4_audit.append({
                "site":site, "width_px":img.width, "height_px":img.height,
                "dpi_x":img.info.get("dpi", (np.nan,np.nan))[0],
                "dpi_y":img.info.get("dpi", (np.nan,np.nan))[1],
                "source_member":zip_member_by_suffix(z, f"{site}_TRACE_PV24_family_permutation_XAI.png"),
                "exported_file":str(source_out.relative_to(ROOT))
            })
        except Exception as e:
            fig4_audit.append({"site":site,"error":str(e)})

fig2_manifest = pd.DataFrame(fig2_manifest)
fig4_audit = pd.DataFrame(fig4_audit)
fig2_manifest.to_csv(OUT_FIG / "Figure2_revision_manifest.csv", index=False)
fig4_audit.to_csv(OUT_FIG / "Figure4_source_resolution_audit.csv", index=False)
display(fig2_manifest)
display(fig4_audit)

## 4. Original TRACE core loader

The next cell loads only the required definition cells from the **guarded TRACE notebook used for the submitted experiment** in the archive. It supports two tasks:

1. Re-export local SHAP plots from the saved model bundles and identical test features.
2. Reuse the original **chronological train → validation selection → train+validation refit → held-out test** workflow for the selection-weight sensitivity analysis.

This does not rerun the complete deep-baseline or global-XAI experiment.

In [ ]:
# ============================================================
# Extract original TRACE core notebook + datasets and load core definitions
# ============================================================
NOTEBOOK_NAME = "TRACE_PV24_Guarded_Dual_Strategy_Ensemble_Windows_Jupyter.ipynb"
DATA_FILES = ["Dangjin_Landfill_PV_Dataset.csv", "Gwangyang_Port_Site2_PV_Dataset.csv"]


def extract_original_core():
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        for base in [NOTEBOOK_NAME, *DATA_FILES]:
            src = zip_member_by_suffix(z, base)
            (WORK_DIR/base).write_bytes(z.read(src))
    return WORK_DIR / NOTEBOOK_NAME


def locate_original_cells(nb):
    markers = {
        "setup":"class Config",
        "data":"REQUIRED_COLUMNS =",
        "features":"class MultiStepPack",
        "models":"def regression_metrics",
        "run":"class Result",
    }
    found = {}
    for i,c in enumerate(nb["cells"]):
        if c.get("cell_type") != "code":
            continue
        src = "".join(c.get("source", []))
        for k,m in markers.items():
            if m in src and k not in found:
                found[k] = (i,src)
    missing = [k for k in markers if k not in found]
    if missing:
        raise RuntimeError(f"Original notebook structure changed; missing markers: {missing}")
    return found

orig_nb_path = extract_original_core()
orig_nb = json.loads(orig_nb_path.read_text(encoding="utf-8"))
ORIG_CELLS = locate_original_cells(orig_nb)
print("Original core cell indices:", {k:v[0] for k,v in ORIG_CELLS.items()})

# Build a reusable namespace. No fitting occurs in these four definition/data cells.
os.environ["TRACE_PV_SKIP_DEEP"] = "1"
os.environ["TRACE_PV_FAST"] = "0"
old_cwd = Path.cwd()
os.chdir(WORK_DIR)
TRACE_NS = {"__name__":"__main__"}
for label in ["setup","data","features","models"]:
    print("Loading original core cell:", label)
    exec(compile(ORIG_CELLS[label][1], f"<TRACE:{label}>", "exec"), TRACE_NS)
os.chdir(old_cwd)
print("Loaded sites:", list(TRACE_NS["PACKS"].keys()))

## 5. Reviewer 1 Comment 5 — local SHAP figure re-export

To address overlapping annotations in Figures 6 and 7, this cell reloads the saved bundle and reduces only the number of displayed waterfall features **without changing the explained observation or fitted TRACE model**.

SHAP's `max_display` parameter controls the number of displayed features. Here it is reduced from 18 to 11 to improve readability.

In [ ]:
# ============================================================
# R1 Comment 5 — re-export local SHAP without retraining
# ============================================================
shap_manifest = []
if RUN_SHAP_REEXPORT:
    try:
        import joblib
        import shap
        import sklearn
        print("Current scikit-learn:", sklearn.__version__)
        if sklearn.__version__ != "1.6.1":
            warnings.warn(
                "Saved sklearn bundles were produced under a 1.6.1-era environment. "
                "For publication re-export, scikit-learn==1.6.1 is preferred."
            )

        with zipfile.ZipFile(ZIP_PATH) as z:
            for site in SITES:
                # Load fitted bundle from original outputs.
                raw = read_bytes_suffix(z, f"{site}_TRACE_PV24_bundle.joblib")
                tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".joblib")
                tmp.write(raw); tmp.close()
                try:
                    bundle = joblib.load(tmp.name)
                finally:
                    Path(tmp.name).unlink(missing_ok=True)

                pred = read_csv_suffix(z, f"{site}_TRACE_PV24_test_predictions.csv")
                pack = TRACE_NS["PACKS"][site]["test"]
                X = pack.X.loc[:, bundle["feature_columns"]].reset_index(drop=True)
                meta = pred.reset_index(drop=True)
                eligible = np.flatnonzero(meta.actual.to_numpy(float) > 0)
                sample_idx = int(eligible[np.argmax(meta.actual.to_numpy(float)[eligible])])
                row = X.iloc[[sample_idx]]

                strategy = bundle["decision"]["strategy"]
                if strategy == "direct_all":
                    targets = [("selected_direct_all", "direct_reg", None)]
                else:
                    targets = [("activity", "extra_cls", 1), ("positive_magnitude", "extra_reg", None)]

                for stage, key, class_index in targets:
                    explainer = shap.TreeExplainer(bundle["models"][key])
                    exp = explainer(row)
                    if exp.values.ndim == 3:
                        exp = shap.Explanation(
                            values=exp.values[:,:,class_index],
                            base_values=np.asarray(exp.base_values)[:,class_index],
                            data=exp.data,
                            feature_names=exp.feature_names,
                        )
                    plt.figure(figsize=SHAP_FIGSIZE)
                    shap.plots.waterfall(exp[0], max_display=SHAP_MAX_DISPLAY, show=False)
                    ax = plt.gca()
                    ax.set_title(
                        f"{site}: local {stage} explanation\n"
                        f"target={meta.target_time.iloc[sample_idx]}",
                        fontsize=15, fontweight="bold", pad=14,
                    )
                    ax.tick_params(axis="both", which="major", labelsize=10)
                    for txt in ax.texts:
                        txt.set_fontsize(SHAP_ANNOTATION_FONTSIZE)
                    plt.tight_layout()
                    out = OUT_SHAP / f"{site}_TRACE_PV24_local_SHAP_{stage}_revised.png"
                    plt.savefig(out, dpi=FIGURE_DPI, bbox_inches="tight", facecolor="white")
                    plt.close()
                    shap_manifest.append({
                        "site":site,"strategy":strategy,"stage":stage,
                        "sample_index":sample_idx,"target_time":meta.target_time.iloc[sample_idx],
                        "actual":float(meta.actual.iloc[sample_idx]),
                        "max_display":SHAP_MAX_DISPLAY,"file":str(out.relative_to(ROOT)),
                    })
    except Exception as e:
        warnings.warn(f"SHAP re-export skipped/failed: {e}")
else:
    print("RUN_SHAP_REEXPORT=False — skipped")

shap_manifest = pd.DataFrame(shap_manifest)
if len(shap_manifest):
    shap_manifest.to_csv(OUT_SHAP / "SHAP_reexport_manifest.csv", index=False)
    display(shap_manifest)

## 6. Reviewer 2 Comment 1 — RMSE/MAE validation-selection weight sensitivity

### What changes?
Only the original experiment's `CFG.selection_mae_weight` is varied.

- Original: MAE weight 0.25 → `0.75 RMSE + 0.25 MAE`
- Alternative 1: 0.50 → `0.50 RMSE + 0.50 MAE`
- Alternative 2: 0.75 → `0.25 RMSE + 0.75 MAE`

### What remains fixed?
- Temporal order and chronological split
- feature generation
- candidate sets
- train → validation selection → train+validation refit → test-once structure
- deep benchmark models

> **Prohibited:** Do not use held-out test RMSE to replace the primary weighting. The alternatives are reported only as reviewer-requested post-hoc robustness checks.

In [ ]:
# ============================================================
# R2 Comment 1 — exact selection-weight sensitivity
# ============================================================
def selected_audit_row(run_dir, site):
    a = pd.read_csv(run_dir / f"{site}_validation_profile_strategy_audit.csv")
    # selected_profile narrows the memory profile; selected_by_guard picks strategy within it.
    s = a[a["selected_profile"].astype(bool) & a["selected_by_guard"].astype(bool)]
    if len(s) != 1:
        sf = a[a["selected_profile"].astype(bool)] if "selected_profile" in a else a
        s = sf.sort_values("robust_selection_score").head(1)
    return s.iloc[0].to_dict()


def result_row(weight, site, result, audit):
    p = result.predictions
    y = p["actual"].to_numpy(float)
    pred = p["pred_TRACE_PV24"].to_numpy(float)
    d = result.decision if isinstance(result.decision, dict) else {}
    return {
        "MAE_weight":weight,
        "RMSE_weight":1-weight,
        "site":site,
        "feature_profile":d.get("feature_profile", audit.get("feature_profile")),
        "strategy":d.get("strategy", audit.get("strategy")),
        "validation_RMSE":audit.get("RMSE"),
        "validation_MAE":audit.get("MAE"),
        "validation_robust_score":audit.get("robust_selection_score"),
        "test_RMSE":rmse(y,pred),
        "test_MAE":mae(y,pred),
        "test_R2":r2(y,pred),
        "alpha_extra_classifier":d.get("weight_extra_classifier", np.nan),
        "beta_extra_regressor":d.get("weight_extra_regressor", np.nan),
        "threshold":d.get("threshold", np.nan),
        "mode":d.get("mode"),
        "source":"reviewer-requested rerun",
    }


def original_baseline_rows():
    rows=[]
    with zipfile.ZipFile(ZIP_PATH) as z:
        final = read_csv_suffix(z, "TRACE_PV24_final_results.csv")
        for site in SITES:
            f = final[final.site==site].iloc[0]
            audit = read_csv_suffix(z, f"{site}_validation_profile_strategy_audit.csv")
            a = audit[audit.selected_profile.astype(bool) & audit.selected_by_guard.astype(bool)]
            if len(a)!=1:
                a = audit[audit.selected_profile.astype(bool)].sort_values("robust_selection_score").head(1)
            a = a.iloc[0]
            rows.append({
                "MAE_weight":0.25,"RMSE_weight":0.75,"site":site,
                "feature_profile":f.feature_profile,"strategy":f.strategy,
                "validation_RMSE":a.RMSE,"validation_MAE":a.MAE,
                "validation_robust_score":a.robust_selection_score,
                "test_RMSE":f.RMSE,"test_MAE":f.MAE,"test_R2":f.R2,
                "alpha_extra_classifier":f.weight_extra_classifier,
                "beta_extra_regressor":f.weight_extra_regressor,
                "threshold":f.threshold,"mode":f["mode"],
                "source":"original submitted experiment",
            })
    return rows

sensitivity_rows = original_baseline_rows() if REUSE_ORIGINAL_025 else []

if RUN_WEIGHT_SENSITIVITY:
    # Use original base CFG loaded from the original notebook.
    base_cfg = TRACE_NS["CFG"]
    old_cwd = Path.cwd()
    os.chdir(WORK_DIR)
    try:
        for w in SENSITIVITY_MAE_WEIGHTS:
            if REUSE_ORIGINAL_025 and abs(w-0.25) < 1e-12:
                continue
            tag = f"rmse{1-w:.2f}_mae{w:.2f}".replace(".","p")
            run_dir = OUT_SENS / tag
            run_dir.mkdir(parents=True, exist_ok=True)
            print(f"\n=== Sensitivity: {(1-w):.2f}*RMSE + {w:.2f}*MAE ===")
            cfg = dataclasses.replace(
                base_cfg,
                selection_mae_weight=float(w),
                output_dir=str(run_dir),
                run_xai=False,
                run_deep_benchmarks=False,
            )
            TRACE_NS["CFG"] = cfg
            TRACE_NS["OUTPUT_DIR"] = run_dir
            # This original cell defines Result/run_site and executes TRACE for the two sites.
            exec(compile(ORIG_CELLS["run"][1], f"<TRACE:sensitivity_w{w:.2f}>", "exec"), TRACE_NS)
            results = TRACE_NS["RESULTS"]
            for site in SITES:
                audit = selected_audit_row(run_dir, site)
                row = result_row(w, site, results[site], audit)
                sensitivity_rows.append(row)
                results[site].predictions.to_csv(
                    run_dir / f"{site}_TRACE_weight_sensitivity_test_predictions.csv", index=False
                )
            pd.DataFrame(sensitivity_rows).to_csv(OUT_SENS / "R2_weight_sensitivity_exact.csv", index=False)
    finally:
        os.chdir(old_cwd)
        TRACE_NS["CFG"] = base_cfg
else:
    print("RUN_WEIGHT_SENSITIVITY=False — only original 0.75/0.25 row will be summarized.")

weight_sensitivity = pd.DataFrame(sensitivity_rows).sort_values(["site","MAE_weight"]).reset_index(drop=True)
weight_sensitivity.to_csv(OUT_SENS / "R2_weight_sensitivity_exact.csv", index=False)
display(weight_sensitivity)

In [ ]:
# ============================================================
# Sensitivity interpretation — validation stability only, never test-based re-selection
# ============================================================
if len(weight_sensitivity):
    stability=[]
    for site, g in weight_sensitivity.groupby("site"):
        stability.append({
            "site":site,
            "weights_available":", ".join(f"{r.RMSE_weight:.2f}/{r.MAE_weight:.2f}" for _,r in g.iterrows()),
            "same_feature_profile_across_weights":g.feature_profile.nunique()==1,
            "same_strategy_across_weights":g.strategy.nunique()==1,
            "feature_profiles":" | ".join(g.feature_profile.astype(str)),
            "strategies":" | ".join(g.strategy.astype(str)),
        })
    sensitivity_stability = pd.DataFrame(stability)
    sensitivity_stability.to_csv(OUT_SENS / "R2_weight_sensitivity_stability.csv", index=False)
    display(sensitivity_stability)

## 7. Consolidated Reviewer action matrix + manuscript insertion text

The final step consolidates the computed evidence, manuscript locations, and reviewer-comment mapping into one action matrix.  
Sensitivity wording is generated only when the corresponding reruns were completed.

In [ ]:
# ============================================================
# Consolidated revision action matrix
# ============================================================
actions = [
    {
        "reviewer_comment":"R1-C1",
        "status":"Text revision only",
        "manuscript_location":"Title; Abstract first TRACE definition; Introduction TRACE definition; Section 3.3 heading",
        "recommended_change":"Rename expansion to Temporal Regime-Aware Chronological Ensemble; use 'Leakage-Safe Features' in Section 3.3; explicitly state no causal-effect identification.",
        "evidence_file":"None required",
    },
    {
        "reviewer_comment":"R1-C2",
        "status":"Solved from stored predictions; no retraining",
        "manuscript_location":"Section 4.4 immediately after Table 9 + Appendix Table A3",
        "recommended_change":"Add solar-eligible and actual-positive benchmark comparison for all main baselines; preserve the Dangjin near-tie wording.",
        "evidence_file":str((OUT_SCOPE/'R1_scope_benchmark_all_models.csv').relative_to(ROOT)),
    },
    {
        "reviewer_comment":"R1-C3",
        "status":"Reporting only; no retraining",
        "manuscript_location":"End of Section 3.4; end of Section 3.5; Appendix A.2/A.4/A.5",
        "recommended_change":"Report candidate ranges, validation-selected settings, lookback/architecture and parameter counts.",
        "evidence_file":str((OUT_SPECS/'R1_neural_model_specs.csv').relative_to(ROOT)),
    },
    {
        "reviewer_comment":"R1-C4",
        "status":"Text revision only",
        "manuscript_location":"Abstract and Conclusion",
        "recommended_change":"Replace 'outperforming 13 baselines' with 'lowest RMSE at both sites, with a near tie at Dangjin and clearer advantage at Gwangyang'.",
        "evidence_file":str((OUT_SCOPE/'R1_scope_benchmark_restricted.csv').relative_to(ROOT)),
    },
    {
        "reviewer_comment":"R1-C5",
        "status":"Visualization-only re-export",
        "manuscript_location":"Figures 6 and 7",
        "recommended_change":f"Use local SHAP waterfall with max_display={SHAP_MAX_DISPLAY}, larger canvas, smaller annotation text; do not change explained observation/model.",
        "evidence_file":str(OUT_SHAP.relative_to(ROOT)),
    },
    {
        "reviewer_comment":"R2-C1",
        "status":"Additional TRACE sensitivity analysis",
        "manuscript_location":"Immediately after Eq. (8) in Section 3.4 + sensitivity table in Results/Appendix A6 + Discussion limitation",
        "recommended_change":"Clarify weighted RMSE/MAE is a validation-selection criterion, not fitting loss; report 0.75/0.25, 0.50/0.50, 0.25/0.75 sensitivity without test-based re-selection.",
        "evidence_file":str((OUT_SENS/'R2_weight_sensitivity_exact.csv').relative_to(ROOT)),
    },
    {
        "reviewer_comment":"R2-C2",
        "status":"Replot stored predictions; no retraining",
        "manuscript_location":"Figure 2 and caption in Section 4.3",
        "recommended_change":"Use complete next-calendar-day profile from 23:00 previous-day origin and state that it is illustrative only, not an operational gate-closure claim.",
        "evidence_file":str((OUT_FIG/'Figure2_revision_manifest.csv').relative_to(ROOT)),
    },
    {
        "reviewer_comment":"R2-C3",
        "status":"Editorial/figure revision only",
        "manuscript_location":"Section 4.4 Figure 3/Table 9; Figure 4",
        "recommended_change":"Keep Table 9 for exact aggregate values; make Figure 3 horizon-wise RMSE only; reinsert original high-resolution Figure 4 source rather than a compressed Word/PDF image.",
        "evidence_file":str((OUT_FIG/'Figure4_source_resolution_audit.csv').relative_to(ROOT)),
    },
]
action_matrix = pd.DataFrame(actions)
action_matrix.to_csv(OUT_SUMMARY / "reviewer_action_matrix.csv", index=False)
display(action_matrix)

In [ ]:
# ============================================================
# Generate manuscript insertion / response evidence markdown
# ============================================================
def f2(x):
    return "NA" if pd.isna(x) else f"{float(x):.2f}"

lines=[]
lines += [
    "# TRACE revision evidence generated by the all-in-one notebook",
    "",
    "## Reviewer 1 Comment 1 — terminology",
    "- Recommended title expansion: **Temporal Regime-Aware Chronological Ensemble**.",
    "- Section 3.3 heading: **Leakage-Safe Features and Structural-Zero Identification**.",
    "- Suggested sentence: 'Here, chronological denotes that every predictor is restricted to information available at the forecast origin; the framework does not perform causal-effect identification.'",
    "",
    "## Reviewer 1 Comment 2 — nighttime-zero fairness",
]
for site in SITES:
    for scope in ["Solar-eligible","Actual-positive"]:
        g = restricted[(restricted.site==site)&(restricted.scope==scope)].sort_values("RMSE")
        tr = g[g.model=="TRACE-PV24"].iloc[0]
        b = g.iloc[0]
        lines.append(f"- {site}, {scope}: TRACE RMSE/MAE = {f2(tr.RMSE)}/{f2(tr.MAE)} kW; lowest-RMSE model = {b.model} ({f2(b.RMSE)} kW).")

lines += [
    "",
    "**Suggested manuscript paragraph after Table 9:**",
    "To assess whether the all-hours benchmark ranking was driven by structurally zero nighttime targets, we recomputed all benchmark metrics over the identical held-out predictions using solar-eligible and actual-positive subsets. The restricted-scope results preserve the main site-level interpretation: Dangjin remains effectively a near tie between TRACE and 1D-CNN–BiLSTM, whereas TRACE retains a clearer advantage at Gwangyang. Thus, the principal benchmark conclusions are not explained solely by the inclusion of structural nighttime zeros.",
    "",
    "## Reviewer 1 Comment 3 — reproducibility",
    "Add Appendix tables for the complete candidate ranges, validation-selected site-specific configurations, and neural architecture/parameter counts. The notebook-generated CSVs in outputs/02_model_specs provide the values.",
    "",
    "## Reviewer 1 Comment 4 — Abstract wording",
    "Suggested wording: **'TRACE obtained the lowest overall RMSE at both sites, with a near tie against 1D-CNN–BiLSTM at Dangjin and a clearer advantage at Gwangyang.'**",
    "",
    "## Reviewer 1 Comment 5 — Figures 6/7",
    f"Re-exported local SHAP waterfall plots use max_display={SHAP_MAX_DISPLAY} with the same explained observation and fitted model.",
    "",
    "## Reviewer 2 Comment 1 — selection-weight sensitivity",
    "**Method clarification to insert immediately after Eq. (8):**",
    "Importantly, L is used only as a validation-based model-selection criterion and is not optimized as the fitting loss. TRACE regressors retain their native fitting objectives, while the neural benchmarks are trained with mean squared error. The RMSE–MAE weighting can therefore affect which candidate configuration is selected, but it does not directly reweight individual training residuals.",
]

if RUN_WEIGHT_SENSITIVITY and weight_sensitivity.MAE_weight.nunique() >= 3:
    lines.append("")
    lines.append("**Sensitivity results generated:**")
    for _,r in weight_sensitivity.iterrows():
        lines.append(
            f"- {r.site}: RMSE/MAE weights {r.RMSE_weight:.2f}/{r.MAE_weight:.2f}; "
            f"profile={r.feature_profile}, strategy={r.strategy}, "
            f"validation RMSE/MAE={f2(r.validation_RMSE)}/{f2(r.validation_MAE)}, "
            f"test RMSE/MAE={f2(r.test_RMSE)}/{f2(r.test_MAE)}."
        )
    for site, g in weight_sensitivity.groupby("site"):
        stable = g.feature_profile.nunique()==1 and g.strategy.nunique()==1
        lines.append(f"- {site} profile/strategy stability across all tested weights: **{stable}**.")
else:
    lines.append("")
    lines.append("Sensitivity alternatives were not executed in this run. Set RUN_WEIGHT_SENSITIVITY=True and Run All before using any robustness claim in the manuscript.")

lines += [
    "",
    "**Do not select a new primary weighting from held-out test performance.** The alternatives are reviewer-requested post-hoc robustness checks.",
    "",
    "## Reviewer 2 Comment 2 — Figure 2",
    "Suggested caption: 'Figure 2 shows an illustrative complete next-calendar-day trajectory issued at 23:00 on the preceding day. The example is used only to visualize an uninterrupted diurnal PV profile and was not used for model selection or performance estimation.'",
    "",
    "## Reviewer 2 Comment 3 — Figure 3/Figure 4",
    "Keep Table 9 for aggregate numerical comparison, use the notebook-generated horizon-RMSE-only Figure 3, and reinsert the original high-resolution family-permutation-XAI source for Figure 4.",
]

summary_md = "\n".join(lines)
(OUT_SUMMARY / "manuscript_insertion_and_response_evidence.md").write_text(summary_md, encoding="utf-8")
display(Markdown(summary_md))

In [ ]:
# ============================================================
# Final manifest
# ============================================================
output_files = [p for p in OUTPUT_ROOT.rglob("*") if p.is_file()]
manifest=[]
for p in sorted(output_files):
    manifest.append({
        "file":str(p.relative_to(ROOT)),
        "bytes":p.stat().st_size,
        "sha256":sha256(p),
    })
manifest = pd.DataFrame(manifest)
manifest.to_csv(OUT_SUMMARY / "generated_output_manifest.csv", index=False)
print(f"Generated {len(manifest)} output files.")
display(manifest)

print("\nDONE")
print("Main summary:", OUT_SUMMARY / "manuscript_insertion_and_response_evidence.md")
print("Reviewer action matrix:", OUT_SUMMARY / "reviewer_action_matrix.csv")
print("Sensitivity results:", OUT_SENS / "R2_weight_sensitivity_exact.csv")